# Optimizing Inference on a Single GPU — Hands-On Exercise

This notebook walks you through **launching** vLLM and SGLang (Modal-supported engines), **running** three prompt types (short, long, multi-turn), and **measuring** prefill, decode, tokens/sec, and GPU usage. You can **compare different Hugging Face models** by changing `MODEL` in §1.2 and re-running.

**Prerequisites:** vLLM and SGLang (both work on Modal). No local GPU required if you use [Modal Notebooks](https://modal.com/notebooks). For gated models: set `HF_TOKEN` in secrets or env.

---
## 1. Setup

### Step 1 — Environment and imports

**Run the cell below once.** You should see: either no output, or one line: `Note: HF_TOKEN not set (optional for Qwen2.5; required for gated models like Llama)`.

In [ ]:
import os
import subprocess
import time
import threading
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from openai import OpenAI

# Optional: load API keys from secrets.env (see secrets.example.env)
try:
    from load_secrets import load_secrets
    load_secrets()
except ImportError:
    pass

# Optional: for results table
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False

# Optional: HF token only needed for gated models (e.g. Llama). Default model Qwen2.5 is non-gated.
if not (os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")):
    print("Note: HF_TOKEN not set (optional for Qwen2.5; required for gated models like Llama)")

### Step 2 — Configuration

**Edit if needed, then run the cell.** Set `MODEL`, `API_HOST` (e.g. `"localhost"` or `"127.0.0.1"` for §1.5), and ports to match your server.

**You should see:** No output. These variables are used by the rest of the notebook.

In [ ]:
# Models to compare (pick one per run, or change MODEL and re-run to compare)
MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",   # small, non-gated
    "Qwen/Qwen2.5-3B-Instruct",     # larger, non-gated
    "microsoft/Phi-3-mini-4k-instruct",  # alternative small model (non-gated)
]
MODEL = MODELS[0]  # change to MODELS[1] or MODELS[2] to benchmark a different model

API_HOST = "localhost"  # or "127.0.0.1" for §1.5 / Modal Notebooks

# Ports (vLLM and SGLang only; run one server at a time per engine)
VLLM_PORT = 8000
SGLANG_PORT = 30000

# Benchmark settings
WARMUP_REQUESTS = 10
N_RUNS = 5
MAX_TOKENS = 128
TEMPERATURE = 0.0
GPU_POLL_INTERVAL_S = 0.5

# Engine flags (when launching servers): vLLM --gpu-memory-utilization 0.9; SGLang --mem-fraction-static 0.85

### Step 2b — (Optional) Download model

If the model is not already cached, uncomment and run once. vLLM/SGLang will also download on first use.

**You should see:** Download progress, or skip this cell if the model is already cached.

In [ ]:
# Optional: pre-download model (skip if already cached)
# from huggingface_hub import snapshot_download
# snapshot_download(MODEL, allow_patterns=["*.safetensors", "*.json"])

### Step 3a — Start a server (terminal)

Start **one** inference server in a **separate terminal** so the notebook can send requests to it. Use the same `MODEL` and port as in §1.2.

**Command line** (use the same `MODEL` as in §1.2):

- **vLLM:** `python -m vllm.entrypoints.openai.api_server --model <MODEL> --host 0.0.0.0 --port 8000`
- **SGLang:** `python -m sglang.launch_server --model-path <MODEL> --host 0.0.0.0 --port 30000`

**Options files:** Copy `vllm_serve_options.example.txt` → `vllm_serve_options.txt` and/or `sglang_serve_options.example.txt` → `sglang_serve_options.txt`, then:
```bash
python launch_server.py --engine vllm --model <MODEL> --options-file vllm_serve_options.txt
python launch_server.py --engine sglang --model <MODEL> --options-file sglang_serve_options.txt
```

Run **one engine at a time** (or use §1.5 for vLLM in-notebook). To compare models, change `MODEL` in §1.2 and restart the server with the new model.

### Step 3b — Start a server from this notebook (Modal Notebooks / no terminal)

If you **only have this notebook** (e.g. uploaded to [Modal Notebooks](https://modal.com/notebooks)):
1. Run **§1.2** first so `MODEL` and `VLLM_PORT` are set.
2. In the cell below, **uncomment** the last line and run it.
3. **You should see:** `Starting vLLM server: model=...`, then `Waiting for server at ...`, then `vLLM server ready after Xs.`
4. Then run **§5** with the vLLM block uncommented and `API_HOST="127.0.0.1"` in §1.2.

**Requirements:** GPU kernel (e.g. A10G), and `vllm` installed (`%pip install vllm` if needed). For SGLang, start it in a separate terminal or run both engines in sequence (start vLLM → run §5 vLLM block → stop vLLM, start SGLang → run §5 SGLang block).

In [ ]:
# Start vLLM server in background (same process as notebook). Run this cell once, then run §5.
import subprocess
import sys

SERVER_PROC = None  # keep reference so process stays alive

def start_vllm_server(model: str = MODEL, host: str = "127.0.0.1", port: int = 8000, timeout_ready: int = 300):
    """Start vLLM OpenAI server in background. Returns the process."""
    global SERVER_PROC
    if SERVER_PROC is not None:
        print("Server already started. Stop it with SERVER_PROC.terminate() if you need to restart.")
        return SERVER_PROC
    print(f"Starting vLLM server: model={model} host={host} port={port} ...", flush=True)
    SERVER_PROC = subprocess.Popen(
        [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", model, "--host", host, "--port", str(port), "--dtype", "bfloat16", "--max-model-len", "8192"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    base_url = f"http://{host}:{port}/v1"
    print(f"Waiting for server at {base_url} (up to {timeout_ready}s)...")
    for i in range(timeout_ready):
        try:
            from openai import OpenAI
            c = OpenAI(base_url=base_url, api_key="x", max_retries=0, timeout=2)
            c.chat.completions.create(model=model, messages=[{"role": "user", "content": "hi"}], max_tokens=2)
            print(f"vLLM server ready after {i}s.")
            return SERVER_PROC
        except Exception:
            if SERVER_PROC.poll() is not None:
                print("Server process exited.")
                break
            if (i + 1) % 15 == 0:
                print(f"  Still waiting... ({i+1}/{timeout_ready})")
        time.sleep(1)
    print("Server did not become ready in time.")
    return None

# Run §1.2 first so MODEL and VLLM_PORT are set. Then uncomment and run:
# start_vllm_server(model=MODEL, host="127.0.0.1", port=VLLM_PORT)
# Then run §5 with the vLLM block uncommented and API_HOST="127.0.0.1" in §1.2.

---
## Step 4 — Client: run inference and measure metrics

**Run the cell below.** It defines how we send requests to the server and measure TTFT, tokens/sec, and total time. No output expected.

- Uses **streaming** when available to measure time to first token (TTFT) and decode time.
- Returns an `InferenceResult` with TTFT, tokens/sec, total time, and (when streaming) prefill_s, decode_s, decode_tokens_per_sec.

In [ ]:
@dataclass
class InferenceResult:
    ttft_s: float | None  # None if non-streaming
    tokens_per_sec: float
    total_time_s: float
    prompt_tokens: int
    completion_tokens: int
    raw_content: str = ""
    prefill_s: float | None = None  # prompt processing until first token (streaming only)
    decode_s: float | None = None   # time from first token to last token (streaming only)
    decode_tokens_per_sec: float | None = None  # completion_tokens / decode_s (streaming only)


def run_inference(
    base_url: str,
    model: str,
    messages: list[dict[str, str]],
    max_tokens: int = 128,
    stream: bool = True,
    temperature: float = 0.0,
    api_key: str = "fake-key",
) -> InferenceResult:
    """
    Send chat completion request and measure TTFT, tokens/sec, total time.
    base_url: e.g. "http://localhost:10210/v1" (no trailing slash).
    temperature=0 for greedy decoding (reproducible benchmarks).
    """
    client = OpenAI(base_url=base_url, api_key=api_key, max_retries=0, timeout=60)
    
    if stream:
        ttft_s: float | None = None
        first_chunk_time: float | None = None
        last_chunk_time: float | None = None
        chunks: list[str] = []
        start = time.perf_counter()
        try:
            stream_obj = client.chat.completions.create(
                model=model,
                messages=messages,
                max_tokens=max_tokens,
                temperature=temperature,
                stream=True,
            )
            for chunk in stream_obj:
                t = time.perf_counter()
                if first_chunk_time is None and chunk.choices and chunk.choices[0].delta.content:
                    first_chunk_time = t
                    ttft_s = first_chunk_time - start
                if chunk.choices and chunk.choices[0].delta.content:
                    last_chunk_time = t
                    chunks.append(chunk.choices[0].delta.content)
        except Exception:
            pass  # fall back to non-streaming below
        
        if first_chunk_time is not None and chunks:
            total_time_s = (last_chunk_time or time.perf_counter()) - start
            content = "".join(chunks)
            completion_tokens = len(chunks)  # approximate if no usage in stream
            tokens_per_sec = completion_tokens / total_time_s if total_time_s > 0 else 0
            prefill_s = ttft_s
            decode_s = (last_chunk_time - first_chunk_time) if last_chunk_time is not None else None
            decode_tokens_per_sec = completion_tokens / decode_s if (decode_s and decode_s > 0) else None
            return InferenceResult(
                ttft_s=ttft_s,
                tokens_per_sec=tokens_per_sec,
                total_time_s=total_time_s,
                prompt_tokens=0,
                completion_tokens=completion_tokens,
                raw_content=content,
                prefill_s=prefill_s,
                decode_s=decode_s,
                decode_tokens_per_sec=decode_tokens_per_sec,
            )
    
    # Non-streaming path (fallback)
    start = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
        stream=False,
    )
    total_time_s = time.perf_counter() - start
    content = resp.choices[0].message.content or ""
    usage = getattr(resp, "usage", None)
    prompt_tokens = usage.prompt_tokens if usage else 0
    completion_tokens = usage.completion_tokens if usage else 0
    tokens_per_sec = completion_tokens / total_time_s if total_time_s > 0 else 0
    return InferenceResult(
        ttft_s=None,
        tokens_per_sec=tokens_per_sec,
        total_time_s=total_time_s,
        prompt_tokens=prompt_tokens,
        completion_tokens=completion_tokens,
        raw_content=content,
        prefill_s=None,
        decode_s=None,
        decode_tokens_per_sec=None,
    )

**Optional:** Run the cell below to define `wait_for_server()`. You can call it before §5 to block until the server is ready. No output until you call it.

In [ ]:
def wait_for_server(base_url: str, model: str, max_retries: int = 30, sleep_s: float = 2.0) -> bool:
    """Poll until server responds. Returns True if ready."""
    client = OpenAI(base_url=base_url, api_key="fake-key", max_retries=0, timeout=5)
    for i in range(max_retries):
        try:
            client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": "hi"}],
                max_tokens=2,
            )
            print(f"Server at {base_url} is ready.")
            return True
        except Exception as e:
            print(f"Waiting for server... ({i + 1}/{max_retries}) {e}")
            time.sleep(sleep_s)
    return False

---
## Step 4b — GPU monitoring

**Run the cell below.** It defines a helper that polls **nvidia-smi** during inference to capture **peak VRAM** and **average GPU utilization %**. No output expected. (On machines without nvidia-smi, GPU stats will be missing in the results.)

In [ ]:
@dataclass
class GPUSnapshot:
    used_mb: float
    util_pct: float


def _poll_nvidia_smi(interval_s: float, stop_event: threading.Event, out_list: list) -> None:
    while not stop_event.is_set():
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used,utilization.gpu", "--format=csv,noheader,nounits"],
                capture_output=True,
                text=True,
                timeout=5,
            )
            if out.returncode == 0 and out.stdout.strip():
                line = out.stdout.strip().split("\n")[0]
                parts = [p.strip() for p in line.split(",")]
                if len(parts) >= 2:
                    used_mb = float(parts[0].split()[0])
                    util = parts[1].strip().replace(" %", "")
                    util_pct = float(util) if util.isdigit() else 0.0
                    out_list.append(GPUSnapshot(used_mb=used_mb, util_pct=util_pct))
        except (FileNotFoundError, subprocess.TimeoutExpired, ValueError):
            pass
        stop_event.wait(interval_s)


def measure_gpu_during_inference(
    inference_fn,
    *args,
    poll_interval_s: float = 0.5,
    **kwargs,
) -> tuple[InferenceResult, float | None, float | None]:
    """
    Run inference_fn(*args, **kwargs) and collect GPU stats via nvidia-smi.
    Returns (result, peak_vram_mb, avg_util_pct).
    """
    kwargs_no_poll = {k: v for k, v in kwargs.items() if k != "poll_interval_s"}
    stop_event = threading.Event()
    snapshots: list[GPUSnapshot] = []

    thread = threading.Thread(target=_poll_nvidia_smi, args=(poll_interval_s, stop_event, snapshots))
    thread.start()
    try:
        result = inference_fn(*args, **kwargs_no_poll)
    finally:
        stop_event.set()
        thread.join(timeout=2)

    peak_vram_mb = max(s.used_mb for s in snapshots) if snapshots else None
    avg_util_pct = (sum(s.util_pct for s in snapshots) / len(snapshots)) if snapshots else None
    return result, peak_vram_mb, avg_util_pct

---
## Step 4c — Prompt scenarios

**Run the cell below.** It defines the three prompt types we benchmark: **short** (quick prefill), **long** (KV cache stress), **multi-turn chat** (prefix caching). No output expected.

In [ ]:
SHORT_PROMPT = [{"role": "user", "content": "What is 2+2? Answer in one word."}]

LONG_PROMPT = [{
    "role": "user",
    "content": "Summarize the following text in 2 sentences. "
    + "The quick brown fox jumps over the lazy dog. " * 80  # ~400+ words
}]

MULTI_TURN_CHAT = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user", "content": "What is one famous landmark there?"},
]

---
## Step 5 — Benchmark loop (definition)

**Run the cell below.** It defines `run_benchmark()`: warm-up, then repeated timed runs per scenario, then mean/p50/p95 for TTFT, prefill, decode, tokens/sec, VRAM, GPU util. No output expected.

**Why warm-up?** Fills CUDA cache and triggers one-time compilations (e.g. CUDA graphs). **Why multiple runs?** Mean/p50/p95 give stable estimates. **Why temperature=0?** Greedy decoding for reproducible comparison. Both vLLM and SGLang support streaming (prefill/decode metrics).

In [ ]:
import numpy as np

def _agg_stats(values: list[float]) -> dict[str, float]:
    """Compute mean, p50, p95 for a list of values (e.g. TTFT or tokens/sec)."""
    a = np.array([v for v in values if v is not None])
    if len(a) == 0:
        return {"mean": float("nan"), "p50": float("nan"), "p95": float("nan")}
    return {
        "mean": float(np.mean(a)),
        "p50": float(np.percentile(a, 50)),
        "p95": float(np.percentile(a, 95)),
    }

def run_benchmark(
    base_url: str,
    model: str,
    engine_name: str,
    stream: bool = False,
    with_gpu: bool = True,
    warmup_requests: int = 10,
    n_runs: int = 5,
    max_tokens: int = 128,
    temperature: float = 0.0,
    poll_interval_s: float = 0.5,
) -> list[dict[str, Any]]:
    """
    Warm-up, then run each scenario n_runs times; aggregate mean/p50/p95 for TTFT, prefill_s, decode_s, decode_tokens_per_sec, tokens/sec, total_time, VRAM, util.
    """
    scenarios = [("short", SHORT_PROMPT), ("long", LONG_PROMPT), ("multi_turn_chat", MULTI_TURN_CHAT)]
    rows: list[dict[str, Any]] = []
    for scenario_name, messages in scenarios:
        # Warm-up: run warmup_requests without timing
        for _ in range(warmup_requests):
            run_inference(base_url, model, messages, max_tokens=max_tokens, stream=stream, temperature=temperature)
        # Timed runs with optional GPU polling
        ttft_list: list[float] = []
        tps_list: list[float] = []
        total_list: list[float] = []
        prefill_list: list[float] = []
        decode_s_list: list[float] = []
        decode_tps_list: list[float] = []
        vram_list: list[float] = []
        util_list: list[float] = []
        for _ in range(n_runs):
            if with_gpu:
                result, peak_vram_mb, avg_util_pct = measure_gpu_during_inference(
                    run_inference, base_url, model, messages,
                    max_tokens=max_tokens, stream=stream, temperature=temperature,
                    poll_interval_s=poll_interval_s,
                )
            else:
                result = run_inference(base_url, model, messages, max_tokens=max_tokens, stream=stream, temperature=temperature)
                peak_vram_mb, avg_util_pct = None, None
            if result.ttft_s is not None:
                ttft_list.append(result.ttft_s)
            if result.prefill_s is not None:
                prefill_list.append(result.prefill_s)
            if result.decode_s is not None:
                decode_s_list.append(result.decode_s)
            if result.decode_tokens_per_sec is not None:
                decode_tps_list.append(result.decode_tokens_per_sec)
            tps_list.append(result.tokens_per_sec)
            total_list.append(result.total_time_s)
            if peak_vram_mb is not None:
                vram_list.append(peak_vram_mb)
            if avg_util_pct is not None:
                util_list.append(avg_util_pct)
        ttft_s = _agg_stats(ttft_list)
        tps_s = _agg_stats(tps_list)
        total_s = _agg_stats(total_list)
        prefill_s = _agg_stats(prefill_list)
        decode_s = _agg_stats(decode_s_list)
        decode_tps_s = _agg_stats(decode_tps_list)
        vram_s = _agg_stats(vram_list)
        util_s = _agg_stats(util_list)
        rows.append({
            "engine": engine_name,
            "streaming": stream,
            "scenario": scenario_name,
            "ttft_mean": round(ttft_s["mean"], 4) if ttft_list else None,
            "ttft_p50": round(ttft_s["p50"], 4) if ttft_list else None,
            "ttft_p95": round(ttft_s["p95"], 4) if ttft_list else None,
            "prefill_s_mean": round(prefill_s["mean"], 4) if prefill_list else None,
            "prefill_s_p50": round(prefill_s["p50"], 4) if prefill_list else None,
            "prefill_s_p95": round(prefill_s["p95"], 4) if prefill_list else None,
            "decode_s_mean": round(decode_s["mean"], 4) if decode_s_list else None,
            "decode_s_p50": round(decode_s["p50"], 4) if decode_s_list else None,
            "decode_s_p95": round(decode_s["p95"], 4) if decode_s_list else None,
            "decode_tok_s_mean": round(decode_tps_s["mean"], 2) if decode_tps_list else None,
            "decode_tok_s_p50": round(decode_tps_s["p50"], 2) if decode_tps_list else None,
            "decode_tok_s_p95": round(decode_tps_s["p95"], 2) if decode_tps_list else None,
            "tokens_per_sec_mean": round(tps_s["mean"], 2),
            "tokens_per_sec_p50": round(tps_s["p50"], 2),
            "tokens_per_sec_p95": round(tps_s["p95"], 2),
            "total_time_s_mean": round(total_s["mean"], 3),
            "total_time_s_p95": round(total_s["p95"], 3),
            "peak_vram_mb_mean": round(vram_s["mean"], 1) if vram_list else None,
            "avg_gpu_util_pct_mean": round(util_s["mean"], 1) if util_list else None,
        })
    return rows

### Step 6 — Run the benchmark

**Run the cell below** (and ensure a server is already running from §1.4 or §1.5).

**You should see:**
- `Benchmark config: <MODEL> warmup=10 n_runs=5 max_tokens=128`
- `Running vLLM benchmark ...` then `vLLM done. 3 rows.`
- `Running SGLang benchmark ...` then `SGLang done. 3 rows.`
- Finally: `Total benchmark rows: 6` (3 scenarios × 2 engines)

If you see a connection error, start the server (§1.4 or §1.5) or check `API_HOST` / ports in §1.2. To compare models, set `MODEL = MODELS[1]` (or another) and restart the server with that model, then re-run §5 and §6.

In [ ]:
# Step 6 — Run the benchmark (vLLM and SGLang)
# Start at least one server first (§1.4 or §1.5). To compare models, set MODEL in §1.2 and restart server with that model.
print("Benchmark config:", MODEL, f"warmup={WARMUP_REQUESTS} n_runs={N_RUNS} max_tokens={MAX_TOKENS}", flush=True)
all_rows: list[dict[str, Any]] = []

# vLLM (streaming)
print("Running vLLM benchmark ...", flush=True)
try:
    base_vllm = f"http://{API_HOST}:{VLLM_PORT}/v1"
    r = run_benchmark(base_vllm, MODEL, "vLLM", stream=True, with_gpu=True,
                     warmup_requests=WARMUP_REQUESTS, n_runs=N_RUNS, max_tokens=MAX_TOKENS,
                     temperature=TEMPERATURE, poll_interval_s=GPU_POLL_INTERVAL_S)
    all_rows.extend(r)
    print("vLLM done.", len(r), "rows.", flush=True)
except Exception as e:
    print(f"vLLM failed: {e}", flush=True)

# SGLang (streaming)
print("Running SGLang benchmark ...", flush=True)
try:
    base_sglang = f"http://{API_HOST}:{SGLANG_PORT}/v1"
    r = run_benchmark(base_sglang, MODEL, "SGLang", stream=True, with_gpu=True,
                     warmup_requests=WARMUP_REQUESTS, n_runs=N_RUNS, max_tokens=MAX_TOKENS,
                     temperature=TEMPERATURE, poll_interval_s=GPU_POLL_INTERVAL_S)
    all_rows.extend(r)
    print("SGLang done.", len(r), "rows.", flush=True)
except Exception as e:
    print(f"SGLang failed: {e}", flush=True)

print("Total benchmark rows:", len(all_rows), flush=True)

---
## Step 7 — View and save results

**Run the cell below.** You should see:
1. A line: `Results: N rows` (e.g. 3 if you ran one engine, 9 if you ran all three).
2. For each scenario (Short, Long, Multi Turn Chat), a table with vLLM and SGLang rows and columns: TTFT, prefill_s, decode_s, tokens_per_sec, total_time_s, VRAM, GPU util.
3. A final line: `Results saved to: .../benchmark_results/benchmark_<timestamp>.md`.

In [ ]:
BENCHMARK_DIR = Path.cwd() / "benchmark_results"  # works locally and on Modal when cwd is class3_runs

def save_benchmark_md(rows: list, config: dict, output_dir: Path) -> Path:
    """Save results to a timestamped .md with config and tables per scenario (matches run_single_gpu_inference.py)."""
    output_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%SZ")
    path = output_dir / f"benchmark_{ts}.md"
    lines = ["# Single-GPU inference benchmark results", "", f"**Run:** {ts}", "", "## Config (flags / settings)", "", "| Option | Value |", "|--------|-------|"]
    for k, v in config.items():
        lines.append(f"| {k} | {v} |")
    lines.extend(["", "## Results", "", "*(Prefill/decode from streaming (vLLM and SGLang).)*", ""])
    if not rows:
        lines.append("(no rows)")
    else:
        def _cell(v):
            return "-" if v is None else str(v)
        def _esc(s):
            return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
        all_keys = set()
        for r in rows:
            all_keys.update(k for k in r.keys() if k != "scenario")
        preferred_order = ["engine", "streaming", "tokens_per_sec_mean", "tokens_per_sec_p50", "tokens_per_sec_p95", "total_time_s_mean", "total_time_s_p95", "peak_vram_mb_mean", "avg_gpu_util_pct_mean", "ttft_mean", "ttft_p50", "ttft_p95", "prefill_s_mean", "prefill_s_p50", "prefill_s_p95", "decode_s_mean", "decode_s_p50", "decode_s_p95", "decode_tok_s_mean", "decode_tok_s_p50", "decode_tok_s_p95"]
        keys = [k for k in preferred_order if k in all_keys]
        keys += sorted(all_keys - set(keys))
        engine_order = ["vLLM", "SGLang"]
        scenario_order = ["short", "long", "multi_turn_chat"]
        by_scenario = {}
        for r in rows:
            sc = r.get("scenario", "")
            by_scenario.setdefault(sc, []).append(r)
        for scenario in scenario_order:
            if scenario not in by_scenario:
                continue
            scenario_rows = sorted(by_scenario[scenario], key=lambda r: (engine_order.index(r.get("engine", "")) if r.get("engine") in engine_order else 99))
            lines.append(f"### {scenario.replace('_', ' ').title()}")
            lines.append("")
            engines = [r.get("engine", "") for r in scenario_rows]
            lines.append("<table>")
            lines.append("<thead><tr><th>Metric</th>" + "".join(f"<th>{_esc(e)}</th>" for e in engines) + "</tr></thead>")
            lines.append("<tbody>")
            for k in keys:
                cells = [_esc(_cell(r.get(k))) for r in scenario_rows]
                lines.append("<tr><td>" + _esc(k) + "</td>" + "".join(f"<td>{c}</td>" for c in cells) + "</tr>")
            lines.append("</tbody></table>")
            lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")
    return path

print(f"Results: {len(all_rows)} rows", flush=True)
if all_rows:
    engine_order = ["vLLM", "SGLang"]
    scenario_order = ["short", "long", "multi_turn_chat"]
    by_scenario = {}
    for r in all_rows:
        by_scenario.setdefault(r.get("scenario", ""), []).append(r)
    for scenario in scenario_order:
        if scenario not in by_scenario:
            continue
        sub = sorted(by_scenario[scenario], key=lambda r: (engine_order.index(r.get("engine", "")) if r.get("engine") in engine_order else 99))
        if HAS_PANDAS:
            df = pd.DataFrame(sub).fillna("-")
            pd.set_option("display.max_columns", None)
            pd.set_option("display.width", None)
            print(f"\n--- {scenario.replace('_', ' ').title()} ---", flush=True)
            try:
                display(df)
            except NameError:
                pass
            print(df.to_string(), flush=True)
        else:
            print(f"\n--- {scenario.replace('_', ' ').title()} ---", flush=True)
            for row in sub:
                print(row, flush=True)
    config = {"engines": "vLLM, SGLang", "model": MODEL, "warmup": str(WARMUP_REQUESTS), "n_runs": str(N_RUNS), "max_tokens": str(MAX_TOKENS), "platform": "notebook"}
    out_path = save_benchmark_md(all_rows, config, BENCHMARK_DIR)
    print(f"\nResults saved to: {out_path}", flush=True)
else:
    print("No results (all_rows is empty).", flush=True)
    print("Run §5 after starting a server: in a terminal use launch_server.py, or on Modal Notebooks run §1.5 to start vLLM in this notebook.", flush=True)

---
## Optional: Flag experimentation

Once you have results from Steps 1–7, you can **change server flags** (via options files or CLI), restart the server, then re-run the benchmark and compare.

**vLLM** (`vllm_serve_options.txt`):
- `--gpu-memory-utilization 0.7` / `0.85` / `0.95` — more → more KV space → higher batch/throughput, OOM risk.
- `--enforce-eager` — disable CUDA graphs (no compile speedups, simpler baseline).
- `--max-num-seqs 128` / `256`.

**SGLang** (`sglang_serve_options.txt`):
- `--mem-fraction-static 0.7` / `0.85` — memory allocation tradeoff.
- `--disable-radix-cache` — baseline without prefix cache; remove to compare with radix cache.
- `--speculative-algorithm EAGLE` (+ draft model path) — if the model supports it (see §8).

In [ ]:
# Example: compare vLLM with different GPU memory (restart server with --gpu-memory-utilization 0.7 vs 0.95)
# r_07 = run_benchmark(f"http://{API_HOST}:{VLLM_PORT}/v1", MODEL, "vLLM (0.7)", stream=True, n_runs=3, warmup_requests=5)
# r_95 = run_benchmark(f"http://{API_HOST}:{VLLM_PORT}/v1", MODEL, "vLLM (0.95)", stream=True, n_runs=3, warmup_requests=5)
# Compare tokens_per_sec_mean, total_time_s_mean across scenarios

---
## Optional: Speculative decoding demo (2–3× decode speedup)

Run the same **creative** prompt with and without speculative decoding; compare **tokens/sec** (often 2–3× higher with spec).

**vLLM:** `--speculative-model <draft> --num-speculative-tokens 5`  
**SGLang:** `--speculative-algorithm EAGLE3 --speculative-draft-model-path <draft>` or `--speculative-algorithm NGRAM`

Start the server once without spec, run `run_spec_demo`; then restart with spec and run again to compare.

In [ ]:
CREATIVE_PROMPT = [{"role": "user", "content": "Write a short poem about coding in exactly 4 lines."}]

def run_spec_demo(base_url: str, model: str, stream: bool = True, n: int = 5) -> dict:
    """Run creative prompt n times (after warm-up), return mean tokens/sec and total_time for comparison."""
    for _ in range(WARMUP_REQUESTS):
        run_inference(base_url, model, CREATIVE_PROMPT, max_tokens=128, stream=stream, temperature=0)
    tps_list = []
    total_list = []
    for _ in range(n):
        r = run_inference(base_url, model, CREATIVE_PROMPT, max_tokens=128, stream=stream, temperature=0)
        tps_list.append(r.tokens_per_sec)
        total_list.append(r.total_time_s)
    out = {"tokens_per_sec_mean": np.mean(tps_list), "total_time_s_mean": np.mean(total_list)}
    print(f"tokens/sec (mean): {out['tokens_per_sec_mean']:.2f}, total_time (mean): {out['total_time_s_mean']:.3f}s")
    return out

# Run without speculative decoding, then restart server WITH spec and run again. Compare tokens_per_sec_mean (expect ~2–3× with spec).
# run_spec_demo(f"http://{API_HOST}:{VLLM_PORT}/v1", MODEL)
# run_spec_demo(f"http://{API_HOST}:{SGLANG_PORT}/v1", MODEL)